# By Sage Fox and Nicholas Fox

In [1]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/Python Code/Paper_condensed/scripts/wrtds_python

In [2]:
import os
import pickle
import numpy as np
import pandas as pd
from scipy import stats
import xarray as xr
import multiprocessing as mp
import queue
from pathlib import Path

import rpy2
import rpy2.robjects as ro
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri

%load_ext rpy2.ipython

In [3]:
trim_to_water_year=True
alternate_percentile=False
add_fluxbiases=False
start_date, end_date = "2022-4-1", "2023-3-31"
water_year = (pd.to_datetime(start_date), pd.to_datetime(end_date))
nboot = 20
nkalman = 25

data_path = Path.cwd().parent.parent/"data"
results_path = Path.cwd().parent.parent/"results"/"wrtds"

csv_out_root = results_path/"csv"
fig_out_root = results_path/"fig"
netcdf_out_root = results_path/"netcdf"

In [4]:
# Load all the data
from data_prep2 import *
areas = pd.Series(load_areas(data_path/"watershed_area_stats.csv")) # TODO: Make load_areas output a series
#areas_l = [areas[r] for r in rivers]
chem_dfs, detection_limits, rng_df = load_chem_dfs(data_path/"wq_uncen_all.xlsx")
scaflow = load_flow_df(data_path/"Downstream_Estimate_All_Rivers_rolling60.csv", start_date, end_date)
unscaflow = load_flow_df(data_path/"unscaled_flow.csv", start_date, end_date)

In [5]:
# Fix Cisnes, which is the only one with duplicate dates but does not have b.d. values
Cisnes_chem = chem_dfs["Cisnes"]
Cisnes_chem = Cisnes_chem.groupby(Cisnes_chem.index).mean(numeric_only=True)
Cisnes_chem["Date"] = Cisnes_chem.index.astype("str")
chem_dfs["Cisnes"] = Cisnes_chem

In [6]:
assert chemicals == ["TN", "TP", "Fe", "dSi", "TSS"]
assert rivers == ["Puelo", "Yelcho", "Palena", "Cisnes", "Aysen"]

In [7]:
# %R install.packages(c("EGRET", "lubridate", "zoo", "EGRETci", "dataRetrieval", "spam", "fields"))
if IN_COLAB:
    %R .libPaths("library_folder")

In [8]:
%%R
library(EGRET)
library(lubridate)
library(zoo)
library(EGRETci)
library(dataRetrieval)
library(spam)
library(fields)

source("modded_Rfuncs.R")
source("wrtds_process.R")
# package_names = ('here', 'tidyverse', 'stringr', 'lubridate', 'reshape2', 'data.table',
#                  'dataRetrieval', 'EGRET', 'EGRETci', 'zoo', 'glue', 'hash')
# package_names = ('here', 'reshape2', 'dataRetrieval', 'EGRET', 'EGRETci', 'zoo', 'hash')

EGRET 3.0.11
Extended Documentation: https://doi-usgs.github.io/EGRET

Attaching package: ‘lubridate’

The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union


Attaching package: ‘zoo’

The following objects are masked from ‘package:base’:

    as.Date, as.Date.numeric

EGRET 2.0.4
Extended Documentation: usgs-r.github.io/EGRETci
dataRetrieval 2.7.23
Extended Documentation: https://doi-usgs.github.io/dataRetrieval
Learn about the new functions that are replacing NWIS functions here:
https://doi-usgs.github.io/dataRetrieval/articles/read_waterdata_functions.html
Consider adding an API_USGS_PAT for new USGS functions.
See: https://api.waterdata.usgs.gov/signup
Spam version 2.11-3 (2026-01-05) is loaded.
Type 'help( Spam)' or 'demo( spam)' for a short introduction 
and overview of this package.
Help for individual functions is also obtained by adding the
suffix '.spam' to the function name, e.g. 'help( chol.spam)'.

Attaching package: ‘spam’

The follow

In [9]:
create_eList = ro.r["create_eList"]
estimate_tables = ro.r["estimate_tables"]

def pd2R(obj):    
    with (ro.default_converter + pandas2ri.converter).context():
        obj_r = ro.conversion.py2rpy(obj)
    return obj_r
def R2pd(r_obj):
    with (ro.default_converter + pandas2ri.converter).context():
        obj_py = ro.conversion.rpy2py(r_obj)
    return obj_py

In [10]:
def repack_tables(Rlist):
  assert list(Rlist.names) == ['eList', 'dailyBootOut', 'dayPct', 'ContConc', 'error'] # sanity check in case things get moved around... hopefully not
  repacked = {
      "eList_INFO": R2pd(Rlist[0][0]),
      "eList_Daily": R2pd(Rlist[0][1]),
      "eList_Sample": R2pd(Rlist[0][2]),
      "eList_surfaces": Rlist[0][3], # Already an ndarray
      # TODO: Remove the eList part to save time? Remove it from wrtds_process?

      "dailyBootOut": np.asarray(Rlist[1]), # Already an ndarray

      "dayPct_flux": R2pd(Rlist[2][0]),
      "dayPct_conc": R2pd(Rlist[2][1]),

      "ContConc": R2pd(Rlist[3]),
      "error": R2pd(Rlist[4]).loc["1"] # Converts from df to series
  }
  # Convert the date of all the outputs from number of days to an actual date
  for key in ["eList_Daily", "eList_Sample", "dayPct_flux", "dayPct_conc", "ContConc"]:
    repacked[key]["Date"] = pd.to_datetime(repacked[key]["Date"], unit="D")
    repacked[key].set_index("Date", inplace=True)

  return repacked


In [11]:
# Trial run to get the array sizes
info_df = format_infodf("Aysen", "TN", areas=areas)
wdf = format_wdf("Aysen", "TN", scaflow, chem_dfs, detection_limits, areas)
aysen_flow = scaflow["Aysen"].rename("Q").reset_index()

eList = create_eList(pd2R(info_df), pd2R(aysen_flow), pd2R(wdf))
a = repack_tables(estimate_tables("Aysen", "TN", eList, nboot, nkalman, 5))

Amt b.d.:  0
Amt NA:  0
Outliers found for Aysen - TN: 0
There are 675 data points, and 675 days.

 discharge data jumps from 2022-11-06 to 2022-11-07
 discharge data jumps from 2023-11-05 to 2023-11-06
 Discharge Record is 675 days long, which is 2 years
 First day of the discharge record is 2022-02-25 and last day is 2023-12-31
 The water quality record has 45 samples
 The first sample is from 2022-03-09 and the last sample is from 2023-08-18
 Discharge: Minimum, mean and maximum 134 525 3560
 Concentration: Minimum, mean and maximum 0.067 0.14 0.29
 Percentage of the sample values that are censored is 0 %

/Users/sagefox/miniconda3/envs/uwgda2023/lib/python3.14/site-packages/rpy2/robjects/pandas2ri.py:65: UserWarning: Error while trying to convert the column "drainSqKm". Fall back to string conversion. The error is: <class 'numpy.float64'>
  warnings.warn('Error while trying to convert '



 first step running estCrossVal may take about 1 minute
 estCrossVal % complete:
2 	4 	6 	8 	11 	13 	15 	17 	20 	
22 	24 	26 	28 	31 	33 	35 	37 	40 	
42 	44 	46 	48 	51 	53 	55 	57 	60 	
62 	64 	66 	68 	71 	73 	75 	77 	80 	
82 	84 	86 	88 	91 	93 	95 	97 	
Next step running  estSurfaces with survival regression:
Survival regression (% complete):
0 	1 	2 	3 	4 	5 	6 	7 	8 	9 	10 	
11 	12 	13 	14 	15 	16 	17 	18 	19 	20 	
21 	22 	23 	24 	25 	26 	27 	28 	29 	30 	
31 	32 	33 	34 	35 	36 	37 	38 	39 	40 	
41 	42 	43 	44 	45 	46 	47 	48 	49 	50 	
51 	52 	53 	54 	55 	56 	57 	58 	59 	60 	
61 	62 	63 	64 	65 	66 	67 	68 	69 	70 	
71 	72 	73 	74 	75 	76 	77 	78 	79 	80 	
81 	82 	83 	84 	85 	86 	87 	88 	89 	90 	
91 	92 	93 	94 	95 	96 	97 	98 	99 	
Survival regression: Done

R callback write-console: Boot: 1
  
R callback write-console: 
There are 16 duplicated Sample dates.
  
R callback write-console: made surfaces from boot sample1replicate
  
R callback write-console:      - Kalman index: 1
  
R callback write-console:      - Kalman index: 2
  
R callback write-console:      - Kalman index: 3
  
R callback write-console:      - Kalman index: 4
  
R callback write-console:      - Kalman index: 5
  
R callback write-console:      - Kalman index: 6
  
R callback write-console:      - Kalman index: 7
  
R callback write-console:      - Kalman index: 8
  
R callback write-console:      - Kalman index: 9
  
R callback write-console:      - Kalman index: 10
  
R callback write-console:      - Kalman index: 11
  
R callback write-console:      - Kalman index: 12
  
R callback write-console:      - Kalman index: 13
  
R callback write-console:      - Kalman index: 14
  
R callback write-console:      - Kalman index: 15
  
R callback write-console:      - Kalman

[1] "FLUX BIAS:  0.0535798328957774"

 Root Mean Squared Error in natural log units =  0.283
 Rsquared for natural log of concentration    =  0.328
 Rsquared for natural log of flux             =  0.87
 Standard error of estimate = 28.9 %


/Users/sagefox/miniconda3/envs/uwgda2023/lib/python3.14/site-packages/rpy2/robjects/vectors.py:1064: UserWarning: R object inheriting from "POSIXct" but without attribute "tzone".
  warnings.warn('R object inheriting from "POSIXct" but without '


In [12]:
date_idx = a["ContConc"].to_xarray()["Date"]
assert (np.array([a["dailyBootOut"].shape[0], ]) == len(date_idx)).all()

# daily_results... same as ContConc I believe
all_daily_results = xr.Dataset( # This should be a dataset as the columns have different dtypes
   data_vars={
       col: xr.DataArray(
           data=np.empty((len(rivers), len(chemicals), len(date_idx))),
           coords={"river": rivers, "chem": chemicals, "Date": date_idx},
           dims=["river", "chem", "Date"]
       )
       for col in a['ContConc'].columns # drop("Date") is because Date is an index but is also a column in the results df
   },
)

all_dailyboot = xr.DataArray(
    data=np.empty((len(rivers), len(chemicals), *a["dailyBootOut"].shape)),
    coords={
        "river": rivers, "chem": chemicals,
        "Date": date_idx, # I think this is right
        "trial": np.arange(a["dailyBootOut"].shape[1]), # do we separate by kalman trial or something? i forget
    },
    dims=["river", "chem", "Date", "trial"]
)

all_samples = xr.DataArray( # This one is made from the dataframes created by data_prep
    # Assuming the it has the same date range as date_idx
    data=np.empty((len(rivers), len(chemicals), len(date_idx))),
    coords={
        "river": rivers, "chem": chemicals,
        "Date": date_idx,
    },
    dims=["river", "chem",  "Date"]
)
# all_remarks = xr.DataArray( # real throwaway thing
#     data=np.empty((len(rivers), len(chemicals), len(date_idx)), dtype="str"),
#     coords={
#         "river": rivers, "chem": chemicals,
#         "Date": date_idx,
#     },
#     dims=["river", "chem",  "Date"]
# )

all_dayPctConc = xr.DataArray(
    data=np.empty((len(rivers), len(chemicals), *a["dayPct_conc"].shape)),
    coords={
        "river": rivers, "chem": chemicals,
        "Date": date_idx,
        "cols": a["dayPct_conc"].columns
    },
    dims=["river", "chem", "Date", "cols"]
)

all_errorstats = xr.DataArray(
    data=np.empty((len(rivers), len(chemicals), len(a["error"]))),
    coords={
        "river": rivers, "chem": chemicals,
        "stats": a["error"].index
    },
    dims=["river", "chem", "stats"]
)

In [13]:
# seeds = pd.DataFrame(columns=rivers,
#                      index=chemicals,
#                      data=np.random.randint(100, size=(len(chemicals), len(rivers)))
#                      )
seeds = pd.read_csv(data_path/"seeds.csv").set_index("chemicals")

In [14]:
on_unix = True # on_unix=False is untested

out_q = mp.Manager().Queue() # Fixed thanks to https://stackoverflow.com/a/56118981
done_q = mp.Manager().Queue()

def run_wrtds(args):
    riv, chem, which_flow, set_seed, nboot, nkalman = args
    display(riv, chem, which_flow)

    wdf = format_wdf(riv, chem, scaflow, chem_dfs, detection_limits, areas)
    info_df = format_infodf(riv, chem, areas)
    if which_flow == "unscaflow":
      flow_df = unscaflow[riv].rename("Q").reset_index()
    elif which_flow == "scaflow":
      flow_df = scaflow[riv].rename("Q").reset_index()
    else:
      raise ValueError("which_flow must be 'unscaflow' or 'scaflow'")

    eList = create_eList(pd2R(info_df), pd2R(flow_df), pd2R(wdf))
    Rout = repack_tables(estimate_tables(riv, chem, eList, nboot, nkalman, int(set_seed)))

    Rout["sample"] = wdf.set_index("Date")[chem]

    # Basically a return statement
    # if on_unix:
    #   all_daily_results.loc[{"river": riv, "chem": chem}] = Rout["ContConc"].to_xarray()
    #   all_dailyboot.loc[{"river": riv, "chem": chem}] = Rout["dailyBootOut"]
    #   all_dayPctConc.loc[{"river": riv, "chem": chem}] = Rout["dayPct_conc"]
    #   all_errorstats.loc[{"river": riv, "chem": chem}] = Rout["error"]
    #   all_samples.loc[{"river": riv, "chem": chem}] = wdf.set_index("Date")[chem].reindex(date_idx) # This is weird... use eList_Sample instead?
    # else:
    out_q.put((riv, chem, which_flow, Rout))

    print("Process done")
    done_q.put(riv, chem, which_flow)
    #f.close()

    # According to https://stackoverflow.com/a/17786444, multiprocessing on unix systems already uses shared memory on global variables...

    # TODO: Refactor so it uses yield and all that? show off?

In [15]:
chemicals

['TN', 'TP', 'Fe', 'dSi', 'TSS']

In [16]:
seeds

,Aysen,Cisnes,Palena,Puelo,Yelcho
chemicals,,,,,
dSi,0,5,0,0,0
TN,0,0,0,0,0
TP,0,0,0,0,0
DOC,53,99,0,0,0
Fe,0,7,0,0,0
TSS,0,0,0,0,0


In [17]:
worker_args = list()
for riv in rivers:
  for chem in chemicals:
    for which_flow in ["scaflow"]:#, "unscaflow"]:
      worker_args.append((riv, chem, which_flow, seeds.loc[chem, riv], nboot, nkalman)) # I would use itertools.product, but this is easier

# Trash the output
stdout = sys.stdout
stderr = sys.stderr
f = open(os.devnull, 'w')
sys.stdout = f
sys.stderr = f

# with mp.get_context("fork").Pool(processes=6) as pool: # Todo: make it not fork
#   state = pool.map_async(run_wrtds, worker_args)
# while not state.ready():
#     if not done_q.empty():
#         display(done_q.get())
for args in worker_args:
    run_wrtds(args)

sys.stdout = stdout
sys.stderr = stderr

f.close()

# TODO: Make a progress bar or graphic somehow
# Also... have the main process eat up out_q concurrently by using pool.async_map

'Puelo'

'TN'

'scaflow'

'Puelo'

'TP'

...

'dSi'

'scaflow'

'Aysen'

'TSS'

'scaflow'

In [18]:
# Put all the queue items into the xarrays

while not out_q.empty():
  riv, chem, which_flow, Rout = out_q.get()
  print(riv, chem, which_flow)

  all_daily_results.loc[{"river": riv, "chem": chem}] = Rout["ContConc"].to_xarray()
  all_dailyboot.loc[{"river": riv, "chem": chem}] = Rout["dailyBootOut"]
  all_dayPctConc.loc[{"river": riv, "chem": chem}] = Rout["dayPct_conc"]
  all_errorstats.loc[{"river": riv, "chem": chem}] = Rout["error"]
  #all_samples.loc[{"river": riv, "chem": chem}] = wdf.set_index("Date")[chem].reindex(date_idx) # This is weird... use eList_Sample instead?
  all_samples.loc[{"river": riv, "chem": chem}] = Rout["eList_Sample"]["ConcAve"].reindex(pd.Index(date_idx)) # Hope this is alright

Puelo TN scaflow
Puelo TP scaflow
Puelo Fe scaflow
Puelo dSi scaflow
Puelo TSS scaflow
Yelcho TN scaflow
Yelcho TP scaflow
Yelcho Fe scaflow
Yelcho dSi scaflow
Yelcho TSS scaflow
Palena TN scaflow
Palena TP scaflow
Palena Fe scaflow
Palena dSi scaflow
Palena TSS scaflow
Cisnes TN scaflow
Cisnes TP scaflow
Cisnes Fe scaflow
Cisnes dSi scaflow
Cisnes TSS scaflow
Aysen TN scaflow
Aysen TP scaflow
Aysen Fe scaflow
Aysen dSi scaflow
Aysen TSS scaflow


In [19]:
date_idx

<xarray.DataArray 'Date' (Date: 675)> Size: 5kB
array(['2022-02-25T00:00:00', '2022-02-26T00:00:00', '2022-02-27T00:00:00',
       ..., '2023-12-29T00:00:00', '2023-12-30T00:00:00',
       '2023-12-31T00:00:00'], shape=(675,), dtype='datetime64[s]')
Coordinates:
  * Date     (Date) datetime64[s] 5kB 2022-02-25 2022-02-26 ... 2023-12-31

In [20]:
# Save these files to prevent future reruns
all_daily_results.to_netcdf(netcdf_out_root/"all_daily_results.nc")
all_dailyboot.to_netcdf(netcdf_out_root/"all_dailyboot.nc")
all_dayPctConc.to_netcdf(netcdf_out_root/"all_dayPctConc.nc")
all_errorstats.to_netcdf(netcdf_out_root/"all_errorstats.nc")
all_samples.to_netcdf(netcdf_out_root/"all_samples.nc")

# 